In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader, Dataset, InMemoryDataset
from torch_geometric.nn import GCNConv, global_mean_pool
from torch.nn import Linear

import imageio
import sknw
import numpy as np
import networkx as nx
import pickle

In [ ]:
# Create a simple dataset and dataloader
# Assume you have a list of 4 graphs (adjacency matrices) in your dataset
# Each graph should be represented as a tuple (adjacency_matrix, class_label)

def create_graph_dataset(group):
    data = []
    group_dict = {'ATL': (26, 3), 'Climp': (31, 2), 'Control': (31, 0), 'RTN': (29, 1)}
    for i in range(1, group_dict[group][0]+1):
        skel = imageio.imread(f'/localhome/asa420/MIAL/data/confocal-data/{group}/er_mean_proc/{group.lower()}{i}_proc_skel.png')

        graph = sknw.build_sknw(skel, multi=False, iso=False)
        # adjacency_matrix = nx.adjacency_matrix(graph).todense()
        adjacency_matrix = nx.to_numpy_matrix(graph)

        data.append((adjacency_matrix, group_dict[group][1]))
    return data


atl_data = pickle.load(open('/localhome/asa420/MIAL/data/confocal-data/atl_mean_graphs.pkl', 'rb'))
climp_data = pickle.load(open('/localhome/asa420/MIAL/data/confocal-data/climp_mean_graphs.pkl', 'rb'))
control_data = pickle.load(open('/localhome/asa420/MIAL/data/confocal-data/control_mean_graphs.pkl', 'rb'))
rtn_data = pickle.load(open('/localhome/asa420/MIAL/data/confocal-data/rtn_mean_graphs.pkl', 'rb'))


graphs = atl_data + climp_data + control_data + rtn_data

dataset = []



for adjacency_matrix, class_label in graphs:
    edge_index = torch.tensor(adjacency_matrix.nonzero(), dtype=torch.long)
    x = torch.ones(adjacency_matrix.shape[0], 1)  # Node features (e.g., all ones)
    data = Data(x=x, edge_index=edge_index, y=torch.tensor([class_label]))
    # data = Data(x=x, edge_index=edge_index, y=torch.tensor([class_label]).unsqueeze(0))  # Ensure y has shape (1,)
    dataset.append(data)


class ERDataset(Dataset):
    def __init__(self, data_list):
        super().__init__()
        self.data_list = data_list

    def len(self):
        return len(self.data_list)

    def get(self, idx):
        return self.data_list[idx]


dataset = ERDataset(dataset)

print()
print(f'Dataset: {dataset}:')
print('====================')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of features: {dataset.num_features}')
print(f'Number of classes: {dataset.num_classes}')

data = dataset[0]  # Get the first graph object.

print()
print(data)
print('=============================================================')

# Gather some statistics about the first graph.
print(f'Number of nodes: {data.num_nodes}')
print(f'Number of edges: {data.num_edges}')
print(f'Average node degree: {data.num_edges / data.num_nodes:.2f}')
print(f'Has isolated nodes: {data.has_isolated_nodes()}')
print(f'Has self-loops: {data.has_self_loops()}')
print(f'Is undirected: {data.is_undirected()}')

In [ ]:
torch.manual_seed(12345)
dataset = dataset.shuffle()

train_dataset = dataset[:95]
test_dataset = dataset[95:]

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=True)


for step, data in enumerate(train_loader):
    print(f'Step {step + 1}:')
    print('=======')
    print(f'Number of graphs in the current batch: {data.num_graphs}')
    print(data)
    print()

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super(GCN, self).__init__()
        torch.manual_seed(12345)
        self.conv1 = GCNConv(dataset.num_node_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, hidden_channels)
        self.lin = Linear(hidden_channels, dataset.num_classes)

    def forward(self, x, edge_index, batch):
        # 1. Obtain node embeddings 
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = self.conv2(x, edge_index)
        x = x.relu()
        x = self.conv3(x, edge_index)

        # 2. Readout layer
        x = global_mean_pool(x, batch)  # [batch_size, hidden_channels]

        # 3. Apply a final classifier
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin(x)
        
        return x

model = GCN(hidden_channels=64)
print(model)

In [ ]:
model = GCN(hidden_channels=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()

def train():
    model.train()

    for data in train_loader:  # Iterate in batches over the training dataset.
         out = model(data.x, data.edge_index, data.batch)  # Perform a single forward pass.
         loss = criterion(out, data.y)  # Compute the loss.
         loss.backward()  # Derive gradients.
         optimizer.step()  # Update parameters based on gradients.
         optimizer.zero_grad()  # Clear gradients.

def test(loader):
     model.eval()

     correct = 0
     for data in loader:  # Iterate in batches over the training/test dataset.
         out = model(data.x, data.edge_index, data.batch)  
         pred = out.argmax(dim=1)  # Use the class with highest probability.
         correct += int((pred == data.y).sum())  # Check against ground-truth labels.
     return correct / len(loader.dataset)  # Derive ratio of correct predictions.


for epoch in range(1, 171):
    train()
    train_acc = test(train_loader)
    test_acc = test(test_loader)
    print(f'Epoch: {epoch:03d}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}')